# Stage 6: Semantic Search (Duplicate Detection)

In this notebook, we embed our training dataset using SentenceTransformers and use it as a retrieval corpus to find similar issues.

In [1]:
import json
import sys
from pathlib import Path

sys.path.append("../src")
from issue_intelligence.models.retrieval import IssueRetriever

DATA_DIR = Path("../data/processed")


## 1. Load Data

In [2]:
def load_data(file_path):
    X = []
    if not file_path.exists():
        return X
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            X.append(r.get("combined_text", ""))
    return X

corpus = load_data(DATA_DIR / "scikit-learn_issues_model_stratified_train.jsonl")
test_queries = load_data(DATA_DIR / "scikit-learn_issues_model_stratified_test.jsonl")

print(f"Corpus size: {len(corpus)}")
print(f"Test queries size: {len(test_queries)}")


Corpus size: 69
Test queries size: 16


## 2. Initialize Retriever and Embed Corpus

In [3]:
retriever = IssueRetriever(model_name="all-MiniLM-L6-v2")
print("Embedding corpus... (This might take a moment the first time)")
retriever.embed_corpus(corpus)
print("Corpus embedded successfully!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding corpus... (This might take a moment the first time)
Corpus embedded successfully!


## 3. Test Retrieval

In [4]:
# Pick a sample query from the test set
sample_query = test_queries[0]
print("QUERY:")
print("-" * 40)
print(sample_query)
print("-" * 40)

print("\nTOP 3 MOST SIMILAR ISSUES IN CORPUS:")
results = retriever.search(sample_query, top_k=3)
for i, res in enumerate(results):
    print(f"\n[{i+1}] Score: {res['score']:.4f}")
    print(res['text'][:200] + "...")


QUERY:
----------------------------------------
This is a Enhancement report number 63 with words feature request documentation add
----------------------------------------

TOP 3 MOST SIMILAR ISSUES IN CORPUS:

[1] Score: 0.9891
This is a Enhancement report number 69 with words feature request documentation add...

[2] Score: 0.9850
This is a Enhancement report number 77 with words feature request documentation add...

[3] Score: 0.9846
This is a Enhancement report number 57 with words feature request documentation add...
